# Breast Cancer Classification using Logistic Regression, Gradio and Flask

**Dataset:** `Cancer_Data.csv`  
**Model:** Logistic Regression  
**Interfaces:** Gradio and Flask  
**Goal:** Predict the dataset label (`M` = malignant, `B` = benign) from the dataset's numerical features.

> **Educational project only:** This model is not a medical diagnostic tool and must not be used to make healthcare decisions.


## 1. Install the required libraries

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn gradio flask pyngrok

## 2. Upload and read the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select Cancer_Data.csv

import pandas as pd
import numpy as np

csv_name = next(name for name in uploaded.keys() if name.lower().endswith(".csv"))
df = pd.read_csv(csv_name)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumns:")
print(df.columns.tolist())

## 3. Clean the dataset and define X and y

In [ ]:
# Remove ID and the empty extra column if present
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

# Remove rows without a target label
df = df.dropna(subset=["diagnosis"]).copy()

# Convert diagnosis labels: B = 0 (benign), M = 1 (malignant)
df["diagnosis"] = df["diagnosis"].map({"B": 0, "M": 1})
df = df.dropna(subset=["diagnosis"])

X = df.drop(columns=["diagnosis"]).apply(pd.to_numeric, errors="coerce")
y = df["diagnosis"].astype(int)

# Fill any missing feature values with each column's median
X = X.fillna(X.median())

feature_names = X.columns.tolist()
print("Number of features:", len(feature_names))
print("Class counts (0=benign, 1=malignant):")
print(y.value_counts().sort_index())

## 4. Split the data and train Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=5000))
])

model.fit(X_train, y_train)
print("Model training completed.")

## 5. Evaluate the model

In [ ]:
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Benign (B)", "Malignant (M)"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Benign (B)", "Malignant (M)"], cmap="Blues"
)
plt.title("Confusion Matrix")
plt.show()

## 6. Create a prediction helper

In [ ]:
def predict_cancer(values):
    # values must be in the same order as feature_names
    row = pd.DataFrame([values], columns=feature_names)
    prediction = int(model.predict(row)[0])
    probabilities = model.predict_proba(row)[0]
    label = "Malignant (M)" if prediction == 1 else "Benign (B)"
    confidence = float(probabilities[prediction])
    return label, confidence

# Test with one row from the dataset
example_values = X.iloc[0].tolist()
predict_cancer(example_values)

## 7. Build the Gradio interface

In [ ]:
import gradio as gr

# Defaults use the training dataset's median for each feature.
feature_medians = X.median().to_dict()

def gradio_predict(*values):
    label, confidence = predict_cancer(values)
    return f"Predicted dataset label: {label}\\nModel confidence: {confidence:.2%}\\n\\nEducational demonstration only — not a medical diagnosis."

gr_inputs = [
    gr.Number(label=name, value=float(feature_medians[name]))
    for name in feature_names
]

with gr.Blocks() as demo:
    gr.Markdown("# Breast Cancer Classification")
    gr.Markdown("Enter numerical feature values. Defaults are dataset medians. This is for educational use only, not medical diagnosis.")
    output = gr.Textbox(label="Prediction", lines=4)
    predict_button = gr.Button("Predict")
    predict_button.click(fn=gradio_predict, inputs=gr_inputs, outputs=output)

demo.launch(share=True, prevent_thread_lock=True)

## 8. Build the Flask interface

In [ ]:
from flask import Flask, request, render_template_string
from pyngrok import ngrok
from getpass import getpass
import threading
import time

# Enter your ngrok auth token when prompted (get it from your ngrok dashboard).
# If you already configured the token in this runtime, you can skip the next two lines.
ngrok_token = getpass("Enter your ngrok auth token: ")
if ngrok_token.strip():
    ngrok.set_auth_token(ngrok_token.strip())

app = Flask("cancer_logistic_regression")

HTML = """
<!doctype html>
<html>
<head>
  <title>Breast Cancer Classification</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 1000px; margin: 30px auto; padding: 0 18px; }
    .grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 12px; }
    label { display: block; font-size: 14px; margin-bottom: 5px; }
    input { width: 100%; box-sizing: border-box; padding: 8px; }
    button { margin-top: 18px; padding: 10px 18px; cursor: pointer; }
    .result { background: #f1f1f1; padding: 14px; margin: 18px 0; white-space: pre-line; }
    .notice { color: #8a1c1c; }
  </style>
</head>
<body>
  <h1>Breast Cancer Classification</h1>
  <p>Enter the numerical features below. Defaults are dataset medians.</p>
  <p class="notice"><b>Educational demonstration only — not a medical diagnostic tool.</b></p>
  {% if result %}<div class="result">{{ result }}</div>{% endif %}
  <form method="post">
    <div class="grid">
    {% for name in feature_names %}
      <div>
        <label for="{{ name }}">{{ name }}</label>
        <input type="number" step="any" id="{{ name }}" name="{{ name }}" value="{{ defaults[name] }}" required>
      </div>
    {% endfor %}
    </div>
    <button type="submit">Predict</button>
  </form>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    result = ""
    if request.method == "POST":
        try:
            values = [float(request.form[name]) for name in feature_names]
            label, confidence = predict_cancer(values)
            result = (
                f"Predicted dataset label: {label}\\n"
                f"Model confidence: {confidence:.2%}\\n\\n"
                "Educational demonstration only — not a medical diagnosis."
            )
        except (ValueError, KeyError):
            result = "Please enter a valid number for every feature."
    return render_template_string(
        HTML, feature_names=feature_names,
        defaults={name: float(feature_medians[name]) for name in feature_names},
        result=result
    )

def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
time.sleep(2)

public_url = ngrok.connect(5000, "http").public_url
print("Open your Flask app here:", public_url)

## 9. Notes
- `B` means benign and `M` means malignant in this dataset. The code maps them to 0 and 1 for classification.
- The scaler is fitted only on the training data by using a scikit-learn Pipeline.
- The Gradio and Flask forms use all numerical predictor columns, so they contain many input fields.
- If you share this project publicly on GitHub, upload this notebook and `Cancer_Data.csv` together.
- Do not interpret predictions as medical advice.